# 네이버 검색 API 수집 · EDA 스크래치

대시보드와 동일한 `src/` 모듈을 노트북에서 직접 호출해 탐색합니다.
`.env` 에 `NAVER_CLIENT_ID` / `NAVER_CLIENT_SECRET` 이 설정되어 있어야 합니다.

In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from datetime import date, timedelta
from src.credentials import load_credentials
from src.naver_client import NaverClient
from src.collect import collect_search, collect_trend
from src import eda

creds = load_credentials()
print('자격증명 설정됨:', creds.is_complete)
client = NaverClient(creds)

In [ ]:
keywords = ['아이폰', '갤럭시']
verticals = ['뉴스', '블로그', '카페글', '지식iN']
end_d = date.today()
start_d = end_d - timedelta(days=30)

res = collect_search(client, keywords, verticals, size=200,
                     start_d=start_d, end_d=end_d, use_cache=True)
res.df.head()

In [ ]:
eda.summary_counts(res.df)

In [ ]:
texts = (res.df['title'] + ' ' + res.df['description']).tolist()
counter = eda.extract_nouns(texts, extra_stop=set(keywords))
counter.most_common(20)

In [ ]:
trend = collect_trend(client, keywords, start_d, end_d, 'date', '', '', [])
trend.pivot_table(index='period', columns='keyword', values='ratio').plot()